In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

from pybedtools import BedTool

from scipy.stats import binomtest, ttest_ind

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snm3C/hicluster'
donors = sorted(list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str)))
tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
bed_path = f'/u/project/cluo/terencew/reference/hg38_igvf/bed/windows/25kb/GRCh38.25kb.bed'
bins = pd.read_csv(bed_path, sep='\t', header=None)
bins.index = [f'{x}_{y}' for x,y in zip(bins[0], bins[1])]
bins.columns = ['chrom', 'start', 'end']
bins.shape

In [ ]:
bins.head()

In [ ]:
boundaries = pd.read_csv(f'{projdir}/csv/tads/ips_cluster_sig.csv', sep='\t', index_col=0)
boundaries.shape

In [ ]:
boundaries.head()

In [ ]:
boundaries['main_ratio'] = boundaries['IPS_main_boundary'] / boundaries['IPS_main_domain']
boundaries['alt_ratio'] = boundaries['IPS_alt_boundary'] / boundaries['IPS_alt_domain']

In [ ]:
select = ['main_ratio', 'alt_ratio']
boundaries[select].idxmax(axis=1).value_counts()

In [ ]:
boundaries_bins = bins.reindex(boundaries.index)
boundaries_bins.shape

In [ ]:
boundaries_bins.head()

In [ ]:
boundaries_bins.to_csv(f'{projdir}/bed/boundaries/ips_alt.bed', sep='\t', header=None, index=None)

In [ ]:
peaks_path = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/atac/peaks/chromhmm/justin.chromhmm.dedup.bed.gz'
peaks = pd.read_csv(peaks_path, sep='\t', header=None)
peaks.index = [f'{x}_{y}' for x,y in zip(peaks[0], peaks[1])]
peaks.shape

In [ ]:
peaks.head()

In [ ]:
peaks_bt = BedTool.from_dataframe(peaks)
boundaries_bt = BedTool.from_dataframe(boundaries_bins)

In [ ]:
inter = peaks_bt.intersect(boundaries_bt).to_dataframe()
inter.shape

In [ ]:
inter.shape[0] / peaks.shape[0]

In [ ]:
gfile = '/u/project/cluo/terencew/reference/hg38_igvf/GRCh38.autosomal.chromsizes'

np.random.seed(1)

n_iters = 1000
seeds = np.random.randint(1e6, size=n_iters)

def shuffle_boundaries(s):
    shuffled_boundaries = boundaries_bt.shuffle(g=gfile, seed=s)
    count = peaks_bt.intersect(shuffled_boundaries, u=True).count()
    return count

In [ ]:
%%time

tmp_results = []
shuffled_results = []
for s in tmp_order:
    print(s)
    tmp_peaks = peaks[peaks[13] == s]
    tmp_peaks = tmp_peaks[~tmp_peaks.index.duplicated()]

    boundaries_bt = BedTool.from_dataframe(boundaries_bins)
    peaks_bt = BedTool.from_dataframe(tmp_peaks)

    peaks_boundaries = peaks_bt.intersect(boundaries_bt, wa=True).to_dataframe(header=None)
    peaks_boundaries.index = [f'{x}_{y}' for x,y in zip(peaks_boundaries[0], peaks_boundaries[1])]

    with ProcessPoolExecutor(max_workers=8) as executor:
        shuffled = list(tqdm(executor.map(shuffle_boundaries, seeds), total=len(seeds)))
        tmp_results.append(peaks_boundaries.shape[0])
        shuffled_results.append(shuffled)

In [ ]:
for i,_ in enumerate(shuffled_results):
    print(tmp_results[i], np.mean(shuffled_results[i]))

In [ ]:
shuffled_means = [np.mean(x) for x in shuffled_results]

In [ ]:
peaks_counts = pd.DataFrame(index=tmp_order)
peaks_counts['all_peaks'] = peaks[13].value_counts()
peaks_counts['peaks_ol_boundaries'] = tmp_results
peaks_counts['shuffled'] = shuffled_means
peaks_counts['frac'] = peaks_counts['peaks_ol_boundaries'] / peaks_counts['all_peaks']
peaks_counts['cluster'] = peaks_counts.index
peaks_counts.shape

In [ ]:
peaks_counts

In [ ]:
n = peaks_counts.shape[0]
pvals = []
for i in range(n):
    
    obs = peaks_counts['peaks_ol_boundaries'][i]
    exp = peaks_counts['shuffled'][i]
    obs_exp = obs / exp

    total_peaks = peaks_counts['all_peaks'][i]
    p = exp / total_peaks
    pval = binomtest(obs, n=total_peaks, p=p, alternative="two-sided").pvalue
    pvals.append(pval)
peaks_counts['binom_pval'] = pvals

### what about just IPS peaks

In [ ]:
ips_peaks = peaks[peaks[13] == 'IPS']
peaks_bt = BedTool.from_dataframe(ips_peaks)
boundaries_bt = BedTool.from_dataframe(boundaries_bins)
inter = peaks_bt.intersect(boundaries_bt).to_dataframe()
inter.shape[0] / ips_peaks.shape[0]

In [ ]:
ips_peaks

In [ ]:
!date